<span style="color:lightgreen">

# Notebook extra 1

En este notebook se experimenta con los parámetros de transcripción con traducción de whisper.

Covost2 pt-en
</span>

# ST Baseline experiment using Whisper and Europarl-ST (Spanish-English)


In this notebook, we are going to learn how to use the Open AI pre-trained model [Whisper](https://openai.com/index/whisper/) for speech translation on the [Europarl-ST dataset](https://huggingface.co/datasets/tj-solergibert/Europarl-ST) (using Spanish-English speech data).

First, we import some OpenAI source whisper libraries and additional ones (e.g. for computing evaluation figures: WER and BLEU)

In [1]:
import os

print(os.getcwd())
if not os.getcwd().endswith("app"):
    os.chdir("../app")
    print(os.getcwd())


%load_ext autoreload
%autoreload 2
%matplotlib inline

from src.config import Configuration

CONFIG = Configuration()

/home/turbotowerlnx/Documents/Master/TA/TA-Portuguesse-English-ST/notebooks_pt_en
/home/turbotowerlnx/Documents/Master/TA/TA-Portuguesse-English-ST/app


In [2]:
import whisper
from whisper.normalizers import BasicTextNormalizer

from tqdm.notebook import tqdm
import pandas as pd

model = whisper.load_model("base")

Load Europarl-ST Spanish-English test audio dataset

In [3]:
from datasets import load_dataset

raw_datasets = load_dataset("facebook/covost2", CONFIG.trans_code, data_dir=CONFIG.corpus_path)
data = raw_datasets["test"]

print(data)

Dataset({
    features: ['client_id', 'file', 'audio', 'sentence', 'translation', 'id'],
    num_rows: 4023
})


<p style="page-break-after:always;"></p>

<span style="color:lightgreen">

Ahora definimos parámetros para el experimento

</span>

In [4]:

experiments = [
    {
        "name": "Experiment 1: Default parameters",
        "normalize": True,
        "params": {
            "temperature": (0.0, 0.2, 0.4, 0.6, 0.8, 1.0),
            "compression_ratio_threshold": 2.4,
            "logprob_threshold": -1.0,
            "no_speech_threshold": 0.6,
            "condition_on_previous_text": True,
            "initial_prompt": None
        }
    },
    {
        "name": "Experiment 2: Low temperature (more deterministic)",
        "normalize": True,
        "params": {
            "temperature": 0.0,
            "compression_ratio_threshold": 2.4,
            "logprob_threshold": -1.0,
            "no_speech_threshold": 0.6,
            "condition_on_previous_text": True,
            "initial_prompt": None
        }
    },
    {
        "name": "Experiment 3: Lenient thresholds",
        "normalize": True,
        "params": {
            "temperature": (0.0, 0.2, 0.4, 0.6, 0.8, 1.0),
            "compression_ratio_threshold": 3.0,
            "logprob_threshold": -1.5,
            "no_speech_threshold": 0.8,
            "condition_on_previous_text": True,
            "initial_prompt": None
        }
    },
    {
        "name": "Experiment 4: No conditioning on previous text",
        "normalize": True,
        "params": {
            "temperature": (0.0, 0.2, 0.4, 0.6, 0.8, 1.0),
            "compression_ratio_threshold": 2.4,
            "logprob_threshold": -1.0,
            "no_speech_threshold": 0.6,
            "condition_on_previous_text": False,
            "initial_prompt": None
        }
    },
    {
        "name": "Experiment 5: With initial prompt and strict thresholds",
        "normalize": True,
        "params": {
            "temperature": 0.0,
            "compression_ratio_threshold": 2.0,
            "logprob_threshold": -0.5,
            "no_speech_threshold": 0.5,
            "condition_on_previous_text": True,
            "initial_prompt": "This is a European Parliament speech about politics and legislation."
        }
    },

    # WITH NO NORMALIZATION
    {
        "name": "Experiment 1: Default parameters",
        "normalize": False,
        "params": {
            "temperature": (0.0, 0.2, 0.4, 0.6, 0.8, 1.0),
            "compression_ratio_threshold": 2.4,
            "logprob_threshold": -1.0,
            "no_speech_threshold": 0.6,
            "condition_on_previous_text": True,
            "initial_prompt": None
        }
    },
    {
        "name": "Experiment 2: Low temperature (more deterministic)",
        "normalize": False,
        "params": {
            "temperature": 0.0,
            "compression_ratio_threshold": 2.4,
            "logprob_threshold": -1.0,
            "no_speech_threshold": 0.6,
            "condition_on_previous_text": True,
            "initial_prompt": None
        }
    },
    {
        "name": "Experiment 3: Lenient thresholds",
        "normalize": False,
        "params": {
            "temperature": (0.0, 0.2, 0.4, 0.6, 0.8, 1.0),
            "compression_ratio_threshold": 3.0,
            "logprob_threshold": -1.5,
            "no_speech_threshold": 0.8,
            "condition_on_previous_text": True,
            "initial_prompt": None
        }
    },
    {
        "name": "Experiment 4: No conditioning on previous text",
        "normalize": False,
        "params": {
            "temperature": (0.0, 0.2, 0.4, 0.6, 0.8, 1.0),
            "compression_ratio_threshold": 2.4,
            "logprob_threshold": -1.0,
            "no_speech_threshold": 0.6,
            "condition_on_previous_text": False,
            "initial_prompt": None
        }
    },
    {
        "name": "Experiment 5: With initial prompt and strict thresholds",
        "normalize": False,
        "params": {
            "temperature": 0.0,
            "compression_ratio_threshold": 2.0,
            "logprob_threshold": -0.5,
            "no_speech_threshold": 0.5,
            "condition_on_previous_text": True,
            "initial_prompt": "This is a European Parliament speech about politics and legislation."
        }
    }
]

# Display experiments
for exp in experiments:
    print(f"\n{exp['name']}")
    for key, value in exp['params'].items():
        print(f"  {key}: {value}")


Experiment 1: Default parameters
  temperature: (0.0, 0.2, 0.4, 0.6, 0.8, 1.0)
  compression_ratio_threshold: 2.4
  logprob_threshold: -1.0
  no_speech_threshold: 0.6
  condition_on_previous_text: True
  initial_prompt: None

Experiment 2: Low temperature (more deterministic)
  temperature: 0.0
  compression_ratio_threshold: 2.4
  logprob_threshold: -1.0
  no_speech_threshold: 0.6
  condition_on_previous_text: True
  initial_prompt: None

Experiment 3: Lenient thresholds
  temperature: (0.0, 0.2, 0.4, 0.6, 0.8, 1.0)
  compression_ratio_threshold: 3.0
  logprob_threshold: -1.5
  no_speech_threshold: 0.8
  condition_on_previous_text: True
  initial_prompt: None

Experiment 4: No conditioning on previous text
  temperature: (0.0, 0.2, 0.4, 0.6, 0.8, 1.0)
  compression_ratio_threshold: 2.4
  logprob_threshold: -1.0
  no_speech_threshold: 0.6
  condition_on_previous_text: False
  initial_prompt: None

Experiment 5: With initial prompt and strict thresholds
  temperature: 0.0
  compression_

<span style="color:lightgreen">
mute warnings

<span>

In [5]:
import torch
import warnings
import os
import logging

# Set Tensor Core precision for RTX 5070
torch.set_float32_matmul_precision('high')

# Suppress PyTorch Lightning tips and info messages
os.environ['PYTHONWARNINGS'] = 'ignore'
warnings.filterwarnings('ignore')
logging.getLogger('pytorch_lightning').setLevel(logging.ERROR)

# Alternative: suppress all Lightning logs
# os.environ['PYTORCH_LIGHTNING_VERBOSITY'] = '0'

In [6]:
from maikol_utils.print_utils import print_separator, print_color
from evaluate import load

normalizer = BasicTextNormalizer()
bleu_metric = load("sacrebleu")
comet_metric = load('comet')
bleu_scores, comet_scores = {}, {}
experiments_results = {}

for exp in experiments:
    print_separator(exp["name"])
    
    experiments_results[exp["name"]] = {}
    translations = []

    for sample in tqdm(data["file"]):   
        translations.append((model.transcribe(
            sample, 
            language=CONFIG.src_name, 
            task="translate",
            **exp["params"]
        ))['text'])

    data_df = pd.DataFrame(dict(translation=translations, reference=data["translation"], source=data["sentence"]))

    if exp["normalize"]:
        data_df["translation_ready"] = [normalizer(text) for text in data_df["translation"]]
        data_df["reference_ready"] = [normalizer(text) for text in data_df["reference"]]
        data_df["source_ready"] = [normalizer(text) for text in data_df["source"]]
    else:
        data_df["translation_ready"] = data_df["translation"]
        data_df["reference_ready"] = data_df["reference"]
        data_df["source_ready"] = data_df["source"]

    bleu_result = bleu_metric.compute(predictions=data_df["translation_ready"], references=data_df["reference_ready"])
    comet_score = comet_metric.compute(predictions=data_df["translation_ready"], references=data_df["reference_ready"], sources=data_df["source_ready"])

    bleu_scores[exp["name"]] = bleu_result["score"]
    comet_scores[exp["name"]] = comet_score['mean_score']

    print_color(f'BLEU score: {bleu_result["score"]:.1f}', color="cyan")
    print_color(f"COMET: {comet_score['mean_score']:.2%}", color="cyan")

    
    experiments_results[exp["name"]]["translations"] = translations
    experiments_results[exp["name"]]["references"] = data_df["reference"]
    experiments_results[exp["name"]]["sources"] = data_df["source"]


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Lightning automatically upgraded your loaded checkpoint from v1.8.3.post1 to v2.6.0. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../../../../../.cache/huggingface/hub/models--Unbabel--wmt22-comet-da/snapshots/2760a223ac957f30acfb18c8aa649b01cf1d75f2/checkpoints/model.ckpt`


Encoder model frozen.


________________________________________________________________
                Experiment 1: Default parameters                



  0%|          | 0/4023 [00:00<?, ?it/s]

...

💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.


GPU available: True (cuda), used: True


TPU available: False, using: 0 TPU cores


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


BLEU score: 19.2
COMET: 63.37%


<span style="color:lightgreen">

### Best scores

<span>

In [7]:
# Best bleu

best_bleu_exp = max(bleu_scores, key=bleu_scores.get)
best_bleu_score = bleu_scores[best_bleu_exp]

print_separator("Best Experiment by BLEU Score")
print_color(f'Best Experiment: {best_bleu_exp} with BLEU score: {best_bleu_score:.1f}', color="green")

# Best comet
best_comet_exp = max(comet_scores, key=comet_scores.get)
best_comet_score = comet_scores[best_comet_exp]

print_separator("Best Experiment by COMET Score")
print_color(f'Best Experiment: {best_comet_exp} with COMET score: {best_comet_score:.2%}', color="green")

________________________________________________________________
                 Best Experiment by BLEU Score                  

Best Experiment: Experiment 3: Lenient thresholds with BLEU score: 22.9
________________________________________________________________
                 Best Experiment by COMET Score                 

Best Experiment: Experiment 2: Low temperature (more deterministic) with COMET score: 64.21%


'\x1bBest Experiment: Experiment 2: Low temperature (more deterministic) with COMET score: 64.21%\x1b'

<p style="page-break-after:always;"></p>

### Best results bleu

In [8]:
data = pd.DataFrame(dict(translation=experiments_results[best_bleu_exp]["translations"], reference=experiments_results[best_bleu_exp]["references"], source=experiments_results[best_bleu_exp]["sources"]))
pd.set_option('display.max_colwidth', None)
data.head(2)

,translation,reference,source
0,and day of going to the people of the day,Borrow money from people in the village,Pedir dinheiro emprestado às pessoas da aldeia
1,We were at it,Lock them up,Trancá-los


Automatic translations, references and sources are normalized using the Whisper basic text standardisation/normalization module

In [9]:
normalizer = BasicTextNormalizer()

data["translation_clean"] = [normalizer(text) for text in data["translation"]]
data["reference_clean"] = [normalizer(text) for text in data["reference"]]
data["source_clean"] = [normalizer(text) for text in data["source"]]
data.head(2)

,translation,reference,source,translation_clean,reference_clean,source_clean
0,and day of going to the people of the day,Borrow money from people in the village,Pedir dinheiro emprestado às pessoas da aldeia,and day of going to the people of the day,borrow money from people in the village,pedir dinheiro emprestado às pessoas da aldeia
1,We were at it,Lock them up,Trancá-los,we were at it,lock them up,trancá los


### Best comet

In [10]:
data = pd.DataFrame(dict(translation=experiments_results[best_comet_exp]["translations"], reference=experiments_results[best_comet_exp]["references"], source=experiments_results[best_comet_exp]["sources"]))
pd.set_option('display.max_colwidth', None)
data.head(2)

,translation,reference,source
0,and day of going to the people of the day,Borrow money from people in the village,Pedir dinheiro emprestado às pessoas da aldeia
1,We're going to drink.,Lock them up,Trancá-los


In [11]:
data["translation_clean"] = [normalizer(text) for text in data["translation"]]
data["reference_clean"] = [normalizer(text) for text in data["reference"]]
data["source_clean"] = [normalizer(text) for text in data["source"]]
data.head(2)

,translation,reference,source,translation_clean,reference_clean,source_clean
0,and day of going to the people of the day,Borrow money from people in the village,Pedir dinheiro emprestado às pessoas da aldeia,and day of going to the people of the day,borrow money from people in the village,pedir dinheiro emprestado às pessoas da aldeia
1,We're going to drink.,Lock them up,Trancá-los,we re going to drink,lock them up,trancá los


<p style="page-break-after:always;"></p>